# Họ và Tên: Nguyễn Hồng Thanh
# MSSV: 2374802010455

## Câu 1 (2 điểm) Chọn một ảnh bất kỳ (tự đặt tên, ví dụ `pic1.jpg`) và thực hiện:

- Làm mờ ảnh bằng box filter. (0.5 điểm)  
- Áp dụng Laplacian để phát hiện biên. (0.5 điểm)  
- Chuyển đổi ảnh sang ảnh âm bản (negative). (0.5 điểm)  
- Chuyển sang không gian màu HSV và lưu 3 kênh H, S, V riêng biệt.  lưu thành ảnh grayscale tương ứng (`[ten_anh]_L.jpg`, `[ten_anh]_A.jpg`, `[ten_anh]_B.jpg`). (0.5 điểm) (0.5 điểm)


In [ ]:
import cv2
import numpy as np

img = cv2.imread('pic1.jpg')

blur = cv2.blur(img, (5,5))
cv2.imwrite('pic1_blur.jpg', blur)

gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
lap = cv2.Laplacian(gray, cv2.CV_64F)
lap_abs = cv2.convertScaleAbs(lap)
cv2.imwrite('pic1_laplacian.jpg', lap_abs)

negative = 255 - img
cv2.imwrite('pic1_negative.jpg', negative)
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
H, S, V = cv2.split(hsv)
cv2.imwrite('pic1_H.jpg', H)
cv2.imwrite('pic1_S.jpg', S)
cv2.imwrite('pic1_V.jpg', V)

### Câu 2 (4 điểm) Viết một chương trình Python sử dụng OpenCV để tạo menu tương tác cho phép người dùng chọn các kỹ thuật chuyển đổi không gian màu và xử lý ảnh nâng cao từ một danh sách, áp dụng đồng thời cho nhiều ảnh.

### Yêu cầu:

1. Menu gồm:  
* Chuyển sang ảnh xám (Grayscale) (0.5 điểm)  
* Chuyển sang HSV (0.5 điểm)  
* Chuyển sang LAB (0.5 điểm)  
* Cân bằng histogram (0.5 điểm)  
* Adaptive Thresholding (tham số ngẫu nhiên) (0.5 điểm)  
* CLAHE (Contrast Limited Adaptive Histogram Equalization) (0.5 điểm)

2. Chương trình xử lý đồng thời 3 ảnh bất kỳ do sinh viên tự chọn (có thể chọn bằng đường dẫn file hoặc nhập tên ảnh tùy ý). (0.5 điểm)

3. Phím tương ứng để kích hoạt các phương pháp xử lý:  
* G: Grayscale  
* H: HSV  
* L: LAB  
* Q: Histogram Equalization  
* A: Adaptive Threshold  
* C: CLAHE (0.5 điểm)

4. Lưu file kết quả với định dạng: `result_[phương pháp]_[tên ảnh gốc].jpg`  
   Ví dụ: `result_gray_flower.jpg`, `result_clahe_img1.jpg` (0.5 điểm)


In [ ]:
import cv2
import numpy as np
import random

def process_image(img, method):
    name = method['name']
    if name == 'G': 
        img_out = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    elif name == 'H': 
        img_out = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    elif name == 'L': 
        img_out = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    elif name == 'Q': 
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        img_out = cv2.equalizeHist(gray)
    elif name == 'A':  
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        block = random.choice([11, 15, 21])
        C = random.randint(2, 10)
        img_out = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_MEAN_C,
                                        cv2.THRESH_BINARY, block, C)
    elif name == 'C': 
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        img_out = clahe.apply(gray)
    else:
        img_out = img
    return img_out

img_names = [input(f'Nhập tên ảnh thứ {i+1}: ') for i in range(3)]
imgs = [cv2.imread(name) for name in img_names]

print("Chọn phương pháp xử lý:")
print("G: Grayscale\nH: HSV\nL: LAB\nQ: Histogram Equalization\nA: Adaptive Threshold\nC: CLAHE")
method_key = input("Nhập ký tự phương pháp: ").upper()
method = {'name': method_key}

for img, name in zip(imgs, img_names):
    img_out = process_image(img, method)
    out_name = f'result_{method_key.lower()}_{name}'
    cv2.imwrite(out_name, img_out)
    print(f'Đã lưu: {out_name}')

### Câu 3 (4 điểm) Viết một chương trình Python để xử lý 3 ảnh bất kỳ do sinh viên tự chọn.

* Cắt ảnh đầu tiên theo tỉ lệ 80% ở giữa. (0.5 điểm)  
* Xoay ảnh thứ hai 90 độ và lật dọc. (0.5 điểm)  
* Thu nhỏ ảnh thứ ba xuống 1/3 kích thước ban đầu và áp dụng Median Blur với kernel 7x7. (1.5 điểm)  
* Thay đổi độ sáng và độ tương phản ảnh thứ ba theo công thức:

$$
I_{out}(x, y) = \alpha \cdot I_{in}(x, y) + \beta
$$

Trong đó:  

$$
\alpha \in [0.7, 1.8], \quad \beta \in [-40, 40]
$$

Giá trị đầu ra cần được giới hạn trong khoảng [0, 255] bằng công thức:

$$
I_{out}(x, y) = \text{clip}(I_{out}(x, y), 0, 255)
$$


In [ ]:
import cv2
import numpy as np

img_names = [input(f'Nhập tên ảnh thứ {i+1}: ') for i in range(3)]
imgs = [cv2.imread(name) for name in img_names]

h1, w1 = imgs[0].shape[:2]
crop_h, crop_w = int(h1 * 0.8), int(w1 * 0.8)
start_y = (h1 - crop_h) // 2
start_x = (w1 - crop_w) // 2
img1_crop = imgs[0][start_y:start_y+crop_h, start_x:start_x+crop_w]
cv2.imwrite(f'result_crop_{img_names[0]}', img1_crop)

img2_rot = cv2.rotate(imgs[1], cv2.ROTATE_90_CLOCKWISE)
img2_flip = cv2.flip(img2_rot, 0)
cv2.imwrite(f'result_rotate_flip_{img_names[1]}', img2_flip)

h3, w3 = imgs[2].shape[:2]
img3_small = cv2.resize(imgs[2], (w3//3, h3//3))
img3_blur = cv2.medianBlur(img3_small, 7)
cv2.imwrite(f'result_small_blur_{img_names[2]}', img3_blur)

alpha = float(input("Nhập alpha (0.7 - 1.8): "))
beta = int(input("Nhập beta (-40 đến 40): "))
img3_bright = cv2.convertScaleAbs(img3_blur, alpha=alpha, beta=beta)
cv2.imwrite(f'result_bright_contrast_{img_names[2]}', img3_bright)

# Chúc các bạn làm bài may mắn, hi vọng mọi người qua môn tất cả được 10.